# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1, The Anatomy of Growing Content

Claim: pages with rising impressions are structurally different from declining pages, specifically longer (3.2K vs 2.3K words) and ranking slightly better (15.9 vs 16.1 avg position).

**Where does the label come from?**
Growing versus declining comes from a self defined threshold, more than 10% impression change over a 30 day window. That is a reasonable rule, but it is a chosen cutoff applied to a noisy signal. A low impression page can cross plus or minus 10% from a few days of random fluctuation alone. The paper does not report how many labeled pages sit near the threshold versus far from it, so it is hard to know how much of the growing and declining split reflects real momentum versus measurement noise, especially in the long tail.

**Does the validation design support the claim?**
Two groups are compared once, on one snapshot, with no significance testing reported. Large N is offered as reassurance, but N stabilizes small differences, it does not make them causally meaningful. The bigger issue is directionality. Word count and position are relatively static covariates being compared against a dynamic growth or decline label. It is equally plausible that editors invest more words in pages that are already gaining traction as it is that added length caused the growth. The paper's own recommendation, expand thin pages that already earn impressions, assumes the causal arrow runs from word count to growth. The design as described cannot distinguish that from the reverse.



### Finding 3, Click Capture by Position Tier

Claim: weighted CTR (total clicks divided by total impressions per tier) drops sharply as position moves away from the top, 0.423% in Top 3 down to 0.050% in Deep, an 88% falloff.

**Where does the label come from?**
Position tier is a direct bucketing of avg_position into fixed, industry standard bins (Top 3, Page 1, Striking Distance, Page 3-5, Deep), not a modeled or fitted label, which is a strength. But avg_position is itself an average over the reporting window, so a page that bounced between position 3 and position 40 gets the same tier label as a page that sat steadily at position 12. The label treats a volatile page and a stable page as interchangeable, and that matters for a claim specifically about click capture, since an averaged position page's CTR isn't cleanly describing tier behavior in either direction.

**Does the validation design support the claim?**
The finding itself, that CTR compresses by tier, measured once and cross sectionally, is well supported. It is a direct aggregate stat with no modeling assumptions in the way. Where the design stops carrying the weight is the recommendation section, which suggests acting on page one pages to convert visibility into click lift. That is a causal claim, better snippet leads to higher CTR, resting on an observational pattern. Nothing in the design rules out reverse or confounded causation. Pages that already have stronger titles, intent match, or brand recognition may both rank higher and earn more clicks, independent of position itself. The measurement is solid, but the prescription assumes a causal direction the design cannot establish on its own.

### Takeaway

Both questions land on the same throughline. The descriptive, well measured comparisons in the paper are solid. It is the jump from these groups differ to doing this will move a page from one group to the other where the validation design and the recommendation start to diverge.



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


In Week 5, my test set was built by sampling 100,000 rows at random from the full June table, before grouping by page. trend_direction depends on having consecutive daily rows for the same content_hash_id, so a row level random sample breaks that structure. Below I diagnose the size of the problem, then rebuild the test set by sampling at the content_hash_id level instead, keeping every day of each selected page within June. This is the honest version of a time aware split, since it preserves the sequence a time aware feature actually needs.

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)

In [ ]:
from datasets import load_dataset
import pandas as pd
cols_needed = [ 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks',  'gsc_avg_position', 'gsc_data_available']
ds_test_raw = load_dataset("FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-06/data_0.parquet",
    split="train", token=hf_token)
ds_test = ds_test_raw.shuffle(seed=42).select(range(100_000))
df_test_source = ds_test.select_columns(cols_needed).to_pandas()
del ds_test_raw, ds_test

cols_needed = [ 'content_hash_id', 'word_count', 'is_published']
ds_content = load_dataset("FlyRank/internship-warehouse",
    data_files="dim_content.parquet", split="train", token=hf_token)
df_content = ds_content.select_columns(cols_needed).to_pandas()
del ds_content

df_test_source = df_test_source.merge(df_content, on="content_hash_id", how="left", validate="m:1")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
from datasets import load_dataset
import pandas as pd

cols_needed = [ 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks',  'gsc_avg_position', 'gsc_data_available']

ds_train = load_dataset("FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-03/data_0.parquet",
    split="train", token=hf_token)
df_train_source = ds_train.select_columns(cols_needed).to_pandas()
del ds_train

ds_test_raw = load_dataset("FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-06/data_0.parquet",
    split="train", token=hf_token)
ds_test = ds_test_raw.shuffle(seed=42).select(range(100_000))
df_test_source = ds_test.select_columns(cols_needed).to_pandas()
del ds_test_raw, ds_test

cols_needed = [ 'content_hash_id', 'word_count', 'is_published', 'content_updated_date']
ds_content = load_dataset("FlyRank/internship-warehouse",
    data_files="dim_content.parquet", split="train", token=hf_token)
df_content = ds_content.select_columns(cols_needed).to_pandas()
del ds_content

df_train_source = df_train_source.merge(df_content, on="content_hash_id", how="left", validate="m:1")
df_test_source = df_test_source.merge(df_content, on="content_hash_id", how="left", validate="m:1")

df_train_source = df_train_source[df_train_source['gsc_data_available'] == True]
df_test_source = df_test_source[df_test_source['gsc_data_available'] == True]


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
df_train_source['report_date'] = pd.to_datetime(df_train_source['report_date'])
df_train_source['content_updated_date'] = pd.to_datetime(df_train_source['content_updated_date'])
df_test_source['report_date'] = pd.to_datetime(df_test_source['report_date'])
df_test_source['content_updated_date'] = pd.to_datetime(df_test_source['content_updated_date'])


df_train_source['word_count_is_stale_safe'] = (
    df_train_source['content_updated_date'].isna() |
    (df_train_source['content_updated_date'] <= df_train_source['report_date'])
)
df_test_source['word_count_is_stale_safe'] = (
    df_test_source['content_updated_date'].isna() |
    (df_test_source['content_updated_date'] <= df_test_source['report_date'])
)

print(df_train_source['word_count_is_stale_safe'].value_counts(normalize=True))
print(df_test_source['word_count_is_stale_safe'].value_counts(normalize=True))

word_count_is_stale_safe
False    0.820345
True     0.179655
Name: proportion, dtype: float64
word_count_is_stale_safe
True     0.671993
False    0.328007
Name: proportion, dtype: float64


In [ ]:
import numpy as np
#Removing bad data, gsc_avg_position at 0, because most have near 0 impressions and clicks, indicating bad data for high position
#it should start at 1
df_train_source.loc[df_train_source['gsc_avg_position'] == 0, 'gsc_avg_position'] = np.nan
df_test_source.loc[df_test_source['gsc_avg_position'] == 0, 'gsc_avg_position'] = np.nan

df_train = df_train_source[df_train_source['gsc_avg_position'].notna()].copy()
df_test = df_test_source[df_test_source['gsc_avg_position'].notna()].copy()

df_train["CTR"] = (df_train["gsc_clicks"] / df_train["gsc_impressions"]) * 100
df_test["CTR"] = (df_test["gsc_clicks"] / df_test["gsc_impressions"]) * 100


df_train.sort_values(by=['content_hash_id', 'report_date'], ascending=True, inplace=True)
df_test.sort_values(by=['content_hash_id', 'report_date'], ascending=True, inplace=True)

df_train['report_date'] = pd.to_datetime(df_train['report_date'])
df_test['report_date'] = pd.to_datetime(df_test['report_date'])

for d in [df_train, df_test]:
    pos_diff = d.groupby('content_hash_id')['gsc_avg_position'].diff()
    days_diff = d.groupby('content_hash_id')['report_date'].diff().dt.days
    d['trend_direction'] = (pos_diff / days_diff).fillna(0)



df_train.loc[~df_train['word_count_is_stale_safe'], 'word_count'] = np.nan
df_test.loc[~df_test['word_count_is_stale_safe'], 'word_count'] = np.nan

In [ ]:
df_train['pos_bucket'] = pd.cut(df_train['gsc_avg_position'], bins=[0,3,10,20,100,500])
df_test['pos_bucket'] = pd.cut(df_test['gsc_avg_position'], bins=[0,3,10,20,100,500])

In [ ]:
import numpy as np



df_train['word_count'] = df_train.groupby('pos_bucket', observed=True)['word_count'].transform(
    lambda x: x.fillna(x.median())
)

df_test['word_count'] = df_test.groupby('pos_bucket', observed=True)['word_count'].transform(
    lambda x: x.fillna(x.median())
)

for col in ['gsc_impressions', 'gsc_avg_position', 'word_count', 'CTR']:
    df_train[col] = np.log1p(df_train[col])
    df_test[col] = np.log1p(df_test[col])

for col in ['CTR','word_count', 'gsc_impressions', 'gsc_clicks']:
    train_median = df_train[col].median()
    test_median = df_test[col].median()

    df_train[col] = df_train[col].fillna(train_median)
    df_test[col] = df_test[col].fillna(test_median)

df_train['trend_direction'] = np.sign(df_train['trend_direction']) * np.log1p(np.abs(df_train['trend_direction']))
df_test['trend_direction'] = np.sign(df_test['trend_direction']) * np.log1p(np.abs(df_test['trend_direction']))


feature_cols = ["gsc_impressions", "gsc_clicks", "word_count", "CTR", "trend_direction"]


In [ ]:
df_train = df_train[(df_train['is_published'] == True) & (df_train['gsc_avg_position'] <= 100)]
df_test = df_test[(df_test['is_published'] == True) & (df_test['gsc_avg_position'] <= 100)]

In [ ]:


rows_per_page = df_test_source.groupby('content_hash_id').size()
print("Row-level sample, rows per content_hash_id:")
print(rows_per_page.value_counts().head())
print(f"Share of pages with only 1 row: {(rows_per_page == 1).mean():.2%}")

print("\ntrend_direction distribution on the row-sampled test set:")
print((df_test['trend_direction'] == 0).mean(), "share exactly zero")

Row-level sample, rows per content_hash_id:
1    27084
2     2846
3      201
4        9
5        1
Name: count, dtype: int64
Share of pages with only 1 row: 89.86%

trend_direction distribution on the row-sampled test set:
0.9024690977466786 share exactly zero


In [ ]:
print(df_content['content_updated_date'].value_counts().head(5))

content_updated_date
2026-05-20    204409
2026-07-01     38981
2026-02-25     32292
2026-06-01     27712
2026-07-03     18034
Name: count, dtype: int64


In [ ]:


ds_test_raw = load_dataset(
    "FlyRank/internship-warehouse",
    data_files="fact_content_daily_performance/month=2026-06/data_0.parquet",
    split="train", token=hf_token
)

cols_needed = [ 'content_hash_id', 'report_date', 'gsc_impressions', 'gsc_clicks',  'gsc_avg_position', 'gsc_data_available']

df_test_full = ds_test_raw.select_columns(cols_needed).to_pandas()
del ds_test_raw

rng = np.random.RandomState(42)
unique_pages = df_test_full['content_hash_id'].unique()
n_pages_target = 5000
sampled_pages = rng.choice(unique_pages, size=min(n_pages_target, len(unique_pages)), replace=False)

df_test_source_honest = df_test_full[df_test_full['content_hash_id'].isin(sampled_pages)].copy()
del df_test_full

df_test_source_honest = df_test_source_honest.merge(df_content, on="content_hash_id", how="left", validate="m:1")
df_test_source_honest = df_test_source_honest[df_test_source_honest['gsc_data_available'] == True]

print(df_test_source_honest.shape)

(47552, 9)


In [ ]:

df_test_source_honest['report_date'] = pd.to_datetime(df_test_source_honest['report_date'])
df_test_source_honest['content_updated_date'] = pd.to_datetime(df_test_source_honest['content_updated_date'])


df_test_source_honest['word_count_is_stale_safe'] = (
    df_test_source_honest['content_updated_date'].isna() |
    (df_test_source_honest['content_updated_date'] <= df_test_source_honest['report_date'])
)
print(df_test_source_honest['word_count_is_stale_safe'].value_counts(normalize=True))

word_count_is_stale_safe
True     0.68382
False    0.31618
Name: proportion, dtype: float64


In [ ]:


df_test_source_honest.loc[df_test_source_honest['gsc_avg_position'] == 0, 'gsc_avg_position'] = np.nan
df_test_honest = df_test_source_honest[df_test_source_honest['gsc_avg_position'].notna()].copy()

df_test_honest["CTR"] = (df_test_honest["gsc_clicks"] / df_test_honest["gsc_impressions"]) * 100
df_test_honest.sort_values(by=['content_hash_id', 'report_date'], ascending=True, inplace=True)
df_test_honest['report_date'] = pd.to_datetime(df_test_honest['report_date'])

pos_diff = df_test_honest.groupby('content_hash_id')['gsc_avg_position'].diff()
days_diff = df_test_honest.groupby('content_hash_id')['report_date'].diff().dt.days
df_test_honest['trend_direction'] = (pos_diff / days_diff).fillna(0)

df_test_honest['pos_bucket'] = pd.cut(df_test_honest['gsc_avg_position'], bins=[0,3,10,20,100,500])
df_test_honest.loc[~df_test_honest['word_count_is_stale_safe'], 'word_count'] = np.nan
df_test_honest['word_count'] = df_test_honest.groupby('pos_bucket', observed=True)['word_count'].transform(
    lambda x: x.fillna(x.median())
)

for col in ['gsc_impressions', 'gsc_avg_position', 'word_count', 'CTR']:
    df_test_honest[col] = np.log1p(df_test_honest[col])

for col in ['CTR', 'word_count', 'gsc_impressions', 'gsc_clicks']:
    df_test_honest[col] = df_test_honest[col].fillna(train_median)

df_test_honest['trend_direction'] = np.sign(df_test_honest['trend_direction']) * np.log1p(np.abs(df_test_honest['trend_direction']))

df_test_honest = df_test_honest[(df_test_honest['is_published'] == True) & (df_test_honest['gsc_avg_position'] <= np.log1p(100))]

rows_per_page_honest = df_test_honest.groupby('content_hash_id').size()
print("Group-level sample, rows per content_hash_id:")
print(rows_per_page_honest.value_counts().head())
print(f"Share of pages with only 1 row: {(rows_per_page_honest == 1).mean():.2%}")
print(f"\nShare of trend_direction exactly zero: {(df_test_honest['trend_direction'] == 0).mean():.2%}")

Group-level sample, rows per content_hash_id:
30    814
1     188
29    110
2     107
28     83
Name: count, dtype: int64
Share of pages with only 1 row: 7.98%

Share of trend_direction exactly zero: 6.03%


In [ ]:
from sklearn.preprocessing import QuantileTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
import numpy as np
import pandas as pd

feature_cols = ["gsc_impressions", "word_count", "CTR", "trend_direction", "gsc_avg_position"]

df_train_k = df_train[feature_cols]
df_test_k = df_test[feature_cols]

scaler = QuantileTransformer(output_distribution='normal', random_state=42)
X_train_scaled = scaler.fit_transform(df_train_k).astype(np.float32)
X_test_scaled = scaler.transform(df_test_k).astype(np.float32)


pca = PCA(n_components=3,random_state=42)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

rng = np.random.RandomState(42)
sample_idx = rng.choice(len(X_train_pca), size=min(1_000_000, len(X_train_pca)), replace=False)
X_sample = X_train_pca[sample_idx]



km_final = KMeans(n_clusters=3, init='k-means++', n_init=20, random_state=42, max_iter=300)
labels = km_final.fit_predict(X_sample)
score = silhouette_score(X_sample, labels, sample_size=5_000, random_state=42)


In [ ]:
# Score the trained clusters and baseline reason code on the honest test set

from statsmodels.stats.proportion import proportion_confint

X_test_honest_scaled = scaler.transform(df_test_honest[feature_cols]).astype(np.float32)
X_test_honest_pca = pca.transform(X_test_honest_scaled)
test_labels_honest = km_final.predict(X_test_honest_pca)
print("silhouette score for test honest split", silhouette_score(X_test_honest_pca, test_labels_honest, sample_size=5_000, random_state=42))


position_raw_th = np.expm1(df_test_honest['gsc_avg_position'].values)
impressions_raw_th = np.expm1(df_test_honest['gsc_impressions'].values)
clicks_raw_th = np.expm1(df_test_honest['gsc_clicks'].values)
trend_raw_th = np.sign(df_test_honest['trend_direction'].values) * np.expm1(np.abs(df_test_honest['trend_direction'].values))

nan_mask_th = pd.isna(position_raw_th)

impressions_safe_th = np.where(impressions_raw_th <= 0, 1, impressions_raw_th)
clicks_safe_th = np.clip(clicks_raw_th, 0, impressions_safe_th)
ci_low_th, ci_high_th = proportion_confint(clicks_safe_th, impressions_safe_th, alpha=0.10, method='wilson')
interval_width_th = ci_high_th - ci_low_th

low_impressions_mask_th = (~nan_mask_th) & (interval_width_th > 0.15)
low_conf_mask_th = (~nan_mask_th) & (position_raw_th > 100)
healthy_mask_th = (~nan_mask_th) & (position_raw_th <= 10) & (~low_impressions_mask_th)
declining_mask_th = (~nan_mask_th) & (position_raw_th > 10) & (position_raw_th <= 100) & (trend_raw_th > 0) & (~low_impressions_mask_th)

reason_code_th = np.full(len(df_test_honest), 'WEAK_BUT_STABLE', dtype=object)
reason_code_th[nan_mask_th] = 'NO_DATA'
reason_code_th[low_impressions_mask_th] = 'LOW_CONFIDENCE_SIGNAL'
reason_code_th[low_conf_mask_th] = 'LOW_CONFIDENCE_SIGNAL'
reason_code_th[healthy_mask_th] = 'HEALTHY'
reason_code_th[declining_mask_th] = 'DECLINING_UNDERPERFORMER'

df_test_honest = df_test_honest.copy()
df_test_honest['baseline_reason_code'] = reason_code_th
df_test_honest['cluster'] = test_labels_honest

crosstab_honest = pd.crosstab(df_test_honest['cluster'], df_test_honest['baseline_reason_code'], normalize='index')
print(crosstab_honest.round(3))
print(df_test_honest['cluster'].value_counts())

silhouette score for test honest split 0.7069223
baseline_reason_code  DECLINING_UNDERPERFORMER  HEALTHY  \
cluster                                                   
0                                        0.000    0.000   
1                                        0.099    0.233   
2                                        0.094    0.597   

baseline_reason_code  LOW_CONFIDENCE_SIGNAL  WEAK_BUT_STABLE  
cluster                                                       
0                                     1.000            0.000  
1                                     0.601            0.068  
2                                     0.236            0.072  
cluster
1    35262
2     5485
0     5360
Name: count, dtype: int64


### Conclusion

The original Week 5 test set was built by sampling 100,000 rows at random from the full June table, before any grouping by page. That row level sample broke the sequence trend_direction depends on. 88.54% of pages in that test set had only one row, and 90.3% of trend_direction values came out exactly zero as a result. This was not a real property of June content, it was a side effect of the sampling method.

That artifact showed up directly in the baseline comparison, and it still shows up after rebuilding the reason code around the Wilson interval instead of the old impressions cutoff. On the row sampled test set under the new rule, DECLINING_UNDERPERFORMER sits at 1.4% in cluster 1 and 1.3% in cluster 2, both close to what the old rule produced on this same broken split and far below what a page level sample produces. Read on its own, this would again look like the Week 5 finding did not generalize to June, the same wrong conclusion the row level sample produced the first time.

After rebuilding the test set by sampling whole content_hash_id pages instead of individual rows, and keeping every day each sampled page had in June, the picture changes again, in the same direction as before. DECLINING_UNDERPERFORMER recovers to 9.9% in cluster 1 and 9.4% in cluster 2. That's lower than the original train figures of 15.0% and 12.9%, which makes sense since the Wilson interval reclassifies some of what used to read as DECLINING_UNDERPERFORMER into LOW_CONFIDENCE_SIGNAL when the click volume behind the trend isn't statistically stable, but it's a real recovery from the row sampled version's 1.3 to 1.4%, not a collapse. Cluster sizes barely moved between the row sampled and group sampled versions, which shows trend_direction was never doing much of the actual separating work inside the clustering itself, even though it clearly matters for how each cluster lines up against the baseline rule.

The honest conclusion is that the apparent collapse of the decline category in a row level test set is a sampling artifact that holds regardless of which version of the reason code rule is used, it isn't something the old impressions cutoff caused or the Wilson interval fixes. Once the split is rebuilt so trend_direction can be computed the way it was intended, on a full sequence of days per page rather than a random scatter of unconnected rows, DECLINING_UNDERPERFORMER recovers under both rule versions. Cluster 1 still splits across the baseline codes rather than concentrating in one, now with LOW_CONFIDENCE_SIGNAL as the largest single share at roughly 60%, and cluster 0's full overlap with LOW_CONFIDENCE_SIGNAL is unaffected either way, since that cluster's separation never depended on trend_direction in the first place.

This is the direct value of an honest, time aware split over a convenience sample. It did not just change a number, it changed which claim was true, and that held up even after the underlying rule the claim was being checked against was rebuilt from scratch.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [ ]:
clustering_features = set(feature_cols)
baseline_inputs = {'gsc_avg_position', 'gsc_impressions', 'trend_direction'}

overlap = clustering_features & baseline_inputs
print("Features used in BOTH clustering and the baseline rule:", overlap)
print(f"{len(overlap)} of {len(clustering_features)} clustering features are also baseline inputs")

Features used in BOTH clustering and the baseline rule: {'gsc_impressions', 'gsc_avg_position', 'trend_direction'}
3 of 5 clustering features are also baseline inputs


In [ ]:
print(df_content.columns.tolist())
print(df_content['content_updated_date'].isna().sum(), "of", len(df_content), "rows have a null content_updated_date")
print(df_content['content_updated_date'].describe())


['content_hash_id', 'word_count', 'is_published', 'content_updated_date']
0 of 519606 rows have a null content_updated_date
count         519606
unique           242
top       2026-05-20
freq          204409
Name: content_updated_date, dtype: object


In [ ]:

print(df_train[['gsc_clicks', 'gsc_impressions', 'CTR']].corr())

                 gsc_clicks  gsc_impressions       CTR
gsc_clicks         1.000000         0.304544  0.352535
gsc_impressions    0.304544         1.000000  0.186657
CTR                0.352535         0.186657  1.000000


In [ ]:
sample_page = df_train['content_hash_id'].value_counts().index[0]
check = df_train[df_train['content_hash_id'] == sample_page][['report_date', 'gsc_avg_position', 'trend_direction']].head(10)
print(check)

        report_date  gsc_avg_position  trend_direction
261772   2026-03-01          3.363016         0.000000
1609680  2026-03-02          3.487051         1.571372
346431   2026-03-03          3.357351        -1.604702
1689787  2026-03-04          3.388962         0.653450
1412822  2026-03-05          3.311025        -1.169989
1557887  2026-03-06          3.284802        -0.536212
2350479  2026-03-07          3.345616         0.983705
2181728  2026-03-08          3.335994        -0.240383
2774003  2026-03-09          3.355225         0.435509
2947908  2026-03-10          3.433907         1.207600


## Leakage Audit Conclusion

Auditing my own features the way I audited the FlyRank paper surfaced two distinct kinds of risk, and they are not equally serious.

Three of five clustering features, gsc_avg_position, gsc_impressions, and trend_direction, are also direct inputs to the baseline_reason_code rule. This does not involve future information, so it is not leakage in the classic sense, but it does make the Section 2 comparison partially circular. When a cluster lines up closely with a baseline category, that agreement is coming from two methods drawing on overlapping raw numbers, not from two fully independent signals converging on the same answer. I checked trend_direction directly against real rows rather than taking the groupby logic on faith, pulling the first ten days of report_date, gsc_avg_position, and trend_direction for a single content_hash_id. The first row in the sequence shows trend_direction at exactly 0.0, with no prior day to diff against, and every row after that shows a value consistent with the position change from the immediately preceding day, for example a drop from 3.363 to 3.487 producing 1.571, and a recovery from 3.487 to 3.357 producing negative 1.605. This confirms trend_direction only uses the current row and the one immediately before it in time, so its only issue is the shared-input circularity, not any leakage of future data. CTR carries a milder version of the same concern, since it is a deterministic function of gsc_clicks and gsc_impressions, and gsc_impressions is already a separate feature in the model, so impressions is implicitly counted twice.word_count's future-edit leakage is masked and imputed rather than left in raw form; the residual risk is that the imputed value is a rough group proxy, not a real content-quality signal, which dilutes word_count's contribution to the clustering

One caveat on precision: 204,409 of the 519,606 pages in dim_content, 39.3%, share the exact same content_updated_date, 2026-05-20. A single date accounting for that large a share of updates is more consistent with a bulk system event, a migration or re-index, than with 200,000 individual organic content edits happening on one day. This means the 82%/31.6% split should be read as a conservative signal that leakage is real and large, not as a precise measurement of true editing behavior, since some rows flagged unsafe may share a placeholder date rather than a genuine edit timestamp.

I have not yet corrected for this. word_count in train, test, and the honest test split still carries the leakage-flagged values as-is, the group-median imputation by position bucket only ever filled rows where word_count was genuinely missing, it did not touch rows where word_count was present but possibly reflects a future edit. So the 82% of train rows and roughly 32% of test and honest-test rows flagged above are sitting in the model exactly as found, unaddressed. This means any contribution word_count makes to cluster separation on train in particular should be treated as unverified until this is fixed, since most of that column's train-side values could be describing the page as it exists in May 2026 rather than as it existed in March. The fact that word_count varies only narrowly across clusters regardless (2,535 to 2,797, per the Week 5 error analysis) is some reassurance that it isn't driving the structure this model found, but that observation was made without the leakage having been corrected, so it isn't proof either way.

The right next step is the same one described above and not yet done: null out word_count for every row where content_updated_date falls after report_date, in train, test, and the honest test split, then let the existing group-median imputation refill those nulls the same way it already handles genuine missingness, and refit scaling, PCA, and KMeans on the corrected data before treating any word_count-driven pattern as trustworthy.

I would not drop word_count entirely once this is fixed, since it is the only content-depth signal available and removing it would erase that dimension from the model altogether. Until the correction is made, this limitation is disclosed directly: any cluster or baseline pattern that leans on word_count should be read as decision-support based on current content state, not as a verified historical predictor, and the leakage described above remains open, not resolved.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original claim, from Week 5, Section 3:**

"Clustering independently rediscovered the exact low-impression population the baseline's impressions < 10 threshold targets, without ever being told about that threshold."

This was the boldest sentence in my original notebook. It uses "independently" and "rediscovered" to imply that two unrelated methods converged on the same answer from separate evidence, which is the strongest possible form of validation a comparison like this could offer.

**Why this claim goes further than the evidence supports:**

The leakage audit in Section 3 found that gsc_avg_position and gsc_impressions, two of the three inputs to the baseline's LOW_CONFIDENCE_SIGNAL rule, are also two of the five features the clustering model was fit on directly. Cluster 0's full overlap with LOW_CONFIDENCE_SIGNAL is not two methods drawing separate conclusions and arriving at the same place. It is two methods drawing on much of the same raw signal and producing a similar boundary, one with a hard threshold, one with a multi-feature distance metric. That is a real and useful result, but it is not independent rediscovery, and calling it that overstates what the comparison actually shows.

**Update after rebuilding the rule around a Wilson interval:**

The baseline's LOW_CONFIDENCE_SIGNAL logic no longer uses a raw impressions-under-10 cutoff. It now flags a row as low confidence when the Wilson interval on CTR, built from gsc_clicks and gsc_impressions, is wider than 0.15. Cluster 0's overlap with LOW_CONFIDENCE_SIGNAL stayed at 100% under this new rule, on train and both test splits. That is a meaningfully different result than before, since the shared-input concern about gsc_impressions doesn't fully carry over, the rule is no longer reading impressions as a raw count, it's reading a statistical property derived from clicks and impressions together, something the clustering was never given directly. gsc_avg_position remains a shared input on both sides, so this still isn't full independence, but it's a narrower and more defensible overlap than the original claim rested on.

**Rewritten claim, safe language:**

"Cluster 0 shows full observed overlap with the baseline's LOW_CONFIDENCE_SIGNAL category, and that overlap held up after the rule was rebuilt around a Wilson interval on CTR rather than a raw impressions cutoff. This is directional evidence that a distance-based grouping over gsc_impressions, gsc_avg_position, and trend_direction produces a boundary similar to the baseline's low-confidence category under two different formulations of that category. gsc_avg_position remains a shared input between the clustering model and the baseline rule, so this should still be read as measured agreement between two related methods, not as full independent confirmation, but the overlap surviving a change to a statistically distinct definition of low confidence is stronger support than agreement with a single fixed threshold would have been. It is decision-support for treating cluster 0 as a usable proxy for the baseline's low-confidence category, not proof that clustering discovered something the rule-based approach could not have found on its own."

**What changed and why it matters:**

The rewritten version keeps the actual finding, the overlap is real, it is total, and it now holds across two structurally different versions of the baseline rule, but it still stops short of claiming independence, since gsc_avg_position is shared either way. "Independently rediscovered" was a causal-sounding claim about two unrelated processes agreeing, disproven by the original leakage audit. "Observed overlap that persists across a rule rebuild" is the same result stated at the level of confidence the evidence actually supports, and it's a slightly stronger statement than what the Week 6 rewrite could make, since at that point the rule hadn't yet been changed to something less circular with the clustering features.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.